In [ ]:
!pip install sentence-transformers chromadb groq pandas -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

In [ ]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import os


In [ ]:
GROQ_API_KEY="your_api_key"
os.environ["GROQ_API_KEY"]=GROQ_API_KEY
groq_client = Groq(api_key=GROQ_API_KEY)

In [ ]:
ds=pd.read_csv("/content/college_notes.csv")
ds.shape


(15, 4)

In [ ]:
ds.columns.tolist()

['note_id', 'subject', 'topic', 'content']

In [ ]:
ds.head()

,note_id,subject,topic,content
0,N001,Data Engineering,ETL Pipelines,ETL stands for Extract Transform Load. It is t...
1,N002,Data Engineering,SQL Databases,A database is an organized collection of data ...
2,N003,Data Engineering,Data Cleaning,Data cleaning involves fixing or removing inco...
3,N004,Data Engineering,APIs and Data Collection,An API or Application Programming Interface al...
4,N005,Data Engineering,Big Data and PySpark,Big Data refers to extremely large datasets th...


In [ ]:
ds['subject'].value_counts()


,count
subject,
Data Engineering,5
Machine Learning,5
Generative AI,3
Python Programming,2


In [ ]:
ds[['note_id','topic','content']].to_string(index=False)

'note_id                          topic                                                                                                                                                                                                                                                                             content\n   N001                  ETL Pipelines                                                            ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.\n   N002                  SQL Databases                                                                   A database is an organized collection of data stored electronically. SQL or Structured Query Language is used to interact with relational databases. Common SQL commands include SELECT INSERT UPDATE and DELETE.\n   N003                  Data Cleaning             

In [ ]:
ds['content_length']=ds['content'].apply(len)

In [ ]:
ds[['topic','content_length']].to_string(index=False)

'                         topic  content_length\n                 ETL Pipelines             216\n                 SQL Databases             209\n                 Data Cleaning             210\n      APIs and Data Collection             224\n          Big Data and PySpark             242\n           Supervised Learning             255\n              Model Evaluation             240\n           Feature Engineering             236\n                Decision Trees             227\n                 Random Forest             238\n         Large Language Models             226\n            Prompt Engineering             275\nRetrieval Augmented Generation             274\n                Pandas Library             252\n            Data Visualization             249'

In [ ]:
document=ds['content'].tolist()

In [ ]:


ids=[f"note_{row['note_id']}" for row in ds.to_dict('records')]
metadatas=[
    {"subject": row['subject'],"topic": row['topic']}
    for row in ds.to_dict('records')
]
print(f"Total chunks prepared: {len(document)}")
print(f"first document ID: {ids[0]}")
print(f"first document metadata: {metadatas[0]}")
print(f"first 100 chars of doc: {document[0][:100]}")

Total chunks prepared: 15
first document ID: note_N001
first document metadata: {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
first 100 chars of doc: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc


In [ ]:
print("loading embedding model")
embedding_model=SentenceTransformer('all-MiniLM-L6-v2')
print("embedding model loaded")
test_embedding = embedding_model.encode("This is test sentence")
print(test_embedding.shape)
print(test_embedding[:5])

loading embedding model


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding model loaded
(384,)
[ 0.07495026  0.07674191 -0.01143933  0.09722367  0.01962913]


In [ ]:
chroma_client=chromadb.Client()
collection=chroma_client.get_or_create_collection(name='college_notes_rag')
print("chroma db created successfully")
print(f"documents in collection: {collection.count()}")

chroma db created successfully
documents in collection: 0


In [ ]:
print("generating embedding for 15 documents")
embeddings=embedding_model.encode(document, show_progress_bar=True)
print(embeddings.shape)
embedding_list=embeddings.tolist()
collection.add(
    documents=document,
    metadatas=metadatas,
    ids=ids,
    embeddings=embedding_list
)
print(collection.count())

generating embedding for 15 documents


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(15, 384)
15


In [ ]:
def retrieve_relevant_chuncks(question, top_k=3):
  question_embedding=embedding_model.encode(question).tolist()
  results=collection.query(
      query_embeddings=[question_embedding],
      n_results=top_k
  )
  return results

In [ ]:
test_question =" what is machine learning and how does it work in data engineering"
print(test_embedding)

results=retrieve_relevant_chuncks(test_question, top_k=3)
print("top 3 retrieved chunks")

for i, (doc,dist,meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
)):
  print(f"Results {i+1}")
  print(f"Subject: {meta['subject']}")
  print(f"topic: {meta['topic']}")
  print(f"distance: {dist}")
  print(f"content: {doc[:100]}")


[ 7.49502555e-02  7.67419115e-02 -1.14393318e-02  9.72236693e-02
  1.96291320e-02  7.49990577e-03  5.25962673e-02 -1.51912554e-03
  1.05633195e-02  4.59739938e-02  1.04660951e-01 -7.69618452e-02
  2.33884621e-03  2.84159947e-02  2.34545744e-03 -2.78552342e-02
  4.36108746e-02 -5.35113923e-02 -7.77004734e-02  2.51303706e-02
  4.50356416e-02  2.45232433e-02 -2.27372963e-02  2.01834049e-02
  1.77401565e-02  2.19538882e-02 -4.97626364e-02  6.96432637e-03
  9.39383656e-02 -2.56639924e-02 -5.24299890e-02  1.44201396e-02
  3.37429233e-02  6.18972182e-02  1.42109599e-02 -9.18646716e-03
  1.78935789e-02  1.48466332e-02  8.23294278e-03  1.00327842e-02
 -2.78042653e-03 -1.22415647e-01  2.43236218e-02  1.40198795e-02
  3.05844527e-02  9.90021508e-03 -3.11313439e-02  2.37757936e-02
 -6.04552729e-03 -3.89494672e-02 -5.96202835e-02 -5.96039481e-02
 -7.59048015e-02 -3.66356149e-02  6.86025666e-03  2.27700733e-02
  2.02777795e-02  1.02620460e-02  2.50371359e-02  4.11647335e-02
  2.97970022e-03 -2.21210

In [ ]:
def build_context_from_results(results):
  context_parts=[]
  for i, (doc, meta) in enumerate(zip(
      results['documents'][0],
      results['metadatas'][0]
  )):
    chunk_text=f"[Source {i+1}: {meta['subject']} - {meta['topic']}]\n{doc}"
    print("top 3 retrieval chuncks")



In [ ]:
def generate_rag_answer(question, context):
  system_prompt="""You are a helpful academic assistant for engineering students.
  You will be given context retrieved from a college knowledge base, and a students's question

  RULES:
  1. Answer only using the information provided in the context below.
  2. If the answer is not found in context, say exactly.
   "I dont have enough information to answer your question"
  3.do not use your general training knowledge
  4.keep answer clear, accurate, and beginner friendly
  """

  user_prompt="""
      explain big bang theory and is impact on expanding universe
  """

  response = groq_client.chat.completions.create(
      model="llama-3.1-8b-instant",
      messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_prompt}
      ],
      temperature=0.1,
      max_tokens=500
  )

  answer= response.choices[0].message.content

  return answer


print("Rag function defined")




Rag function defined


In [ ]:
def ask_college_assistant(question, top_k=3, verbose=True):
  if verbose:
    print(f"Question: {question}")

  # 1. Retrieve relevant documents
  results = retrieve_relevant_chuncks(question, top_k=top_k)
  if verbose:
    print(f"Retrieved {len(results['documents'][0])} relevant document(s).")

  # 2. Build context from retrieved documents
  context = build_context_from_results(results)
  if verbose:
    print("Context built from relevant documents.")
    print("\n--- Retrieved Context ---")
    print(context)
    print("-------------------------")

  # 3. Generate answer using RAG
  answer = generate_rag_answer(question, context)

  if verbose:
    print("\n--- Generated Answer ---")
    print(answer)
    print("-------------------------")

  return answer